In [1]:
import os
import sys
import numpy as np

module_path = os.path.abspath(os.path.join('..', ''))
if module_path not in sys.path:
    sys.path.append(module_path)

from rect_prune_ood.lib.util.paths import get_top_level_path

results_dir = os.path.join(get_top_level_path(), 'results')
imagenet_c_results_dir = os.path.join(results_dir, 'imagenet', "baseline", "imagenet_c")
ablation_study_results_dir = os.path.join(results_dir, 'ablation_study_pertubation')
os.makedirs(ablation_study_results_dir, exist_ok=True)

imagenet_c_pertubations = [
        "gaussian_noise", 
        "saturate",
        "brightness",
        "pixelate",
        "snow",
        "motion_blur",
        "elastic_transform",
        "fog",
        "zoom_blur",
        "shot_noise",
        "contrast",
        "defocus_blur",
        "frost",
        "impulse_noise",
        "jpeg_compression",
        "glass_blur",
    ]

imagenet_c_pertubations_file = [
        "brightness",
        "defocus_blur",
        "fog",
        "gaussian_blur",
        "glass_blur",
        "jpeg_compression",
        "pixelate",
        "shot_noise",
        "spatter",
        "zoom_blur",
        "contrast",
        "elastic_transform",
        "frost",
        "gaussian_noise", 
        "impulse_noise",
        "motion_blur",
        "saturate",
        "snow",
        "speckle_noise"
    ]

In [2]:
from torch.nn import Module
from torch import Tensor


class APESP(Module):
    def __init__(self, mask) -> None:
        #super(LayerScaler, self).__init__()
        super(APESP, self).__init__()
        self.mask = mask
        
    
    def forward(self, x: Tensor) -> Tensor:
        x = self.apes_p(x)
        return x
    
    def apes_p(self, x):
        assert x.dim() == 4
        _, c, _, _ = x.shape
        assert self.mask.shape[0] == c 

        x = x * self.mask

        return x

In [ ]:
imagenet_c_activations = np.load(os.path.join(imagenet_c_results_dir, "imagenet_c_activations_severity_5.npy"))
imagenet_activations = np.load(os.path.join(results_dir, 'imagenet', 'baseline', 'imagenet_activations.npy'))
imagenet_activations = np.transpose(imagenet_activations)

(50000, 2048)
(2048, 50000)


In [14]:
import torch
from lightning import Trainer
from rect_prune_ood.lib.experiments.feature_extractor import FeatureExtractor
from rect_prune_ood.lib.experiments.backbones import get_resnet50
from rect_prune_ood.lib.experiments.pipeline import Pipeline
from rect_prune_ood.lib.util.data_loader import get_data_loader

masks = []

for i in range(len(imagenet_c_pertubations)):

    if i == 2:
        break

    # pertubation_index = []
    # for j in range(i+1):
    #     pertubation_index.append(imagenet_c_pertubations_file.index(imagenet_c_pertubations[j]))

    # # concatenate the activations. data has shape (19, 2048, 50000), goal is to get (2048, x) 
    # activations = np.concatenate([imagenet_c_activations[j] for j in pertubation_index], axis=1)

    activations = imagenet_c_activations[i]
    # concatenate with imagenet activations
    activations = np.concatenate([activations, imagenet_activations], axis=1)

    variance = np.var(activations, axis=1)
    variance = torch.tensor(variance)

    percentile = np.percentile(variance, 85)

    mask = torch.where(variance <= percentile, 1, 0)
    mask = mask.view(variance.shape[0], 1, 1).cuda()
    
    feature_extractor = FeatureExtractor(get_resnet50(), 'avgpool')
    
    fc_layer = fc_layer=feature_extractor.get_fc_layer()

    pipeline = Pipeline(feature_extractor, 'baseline', fc_layer, caputure_activations=False)
    pipeline.method_list = [APESP(mask)]

    dataloader, _ = get_data_loader('imagenet', batch_size=16)

    # Trainer
    trainer = Trainer(logger=None)
    trainer.test(pipeline, dataloader)

    results = pipeline.results

    np.save(os.path.join(ablation_study_results_dir, f"ablation_study_pertubations_imagenet_{i+1}.npy"), results)

    feature_extractor = FeatureExtractor(get_resnet50(), 'avgpool')
    
    fc_layer = fc_layer=feature_extractor.get_fc_layer()

    pipeline = Pipeline(feature_extractor, 'baseline', fc_layer, caputure_activations=False)
    pipeline.method_list = [APESP(mask)]

    dataloader, _ = get_data_loader('openimage-o', batch_size=16)

    # Trainer
    trainer = Trainer(logger=None)
    trainer.test(pipeline, dataloader)

    results = pipeline.results

    np.save(os.path.join(ablation_study_results_dir, f"ablation_study_pertubations_openimage-o_{i+1}.npy"), results)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

In [25]:
differences = 0

for i in range (2048):
    # compute average mask entry for each mask
    average = 0
    for mask in masks:
        average += mask[i]

    float_average = average / len(masks)

    if float_average != 0 and float_average != 1:
        print(float_average)
        differences += 1

print(differences)

tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.1875)
tensor(0.8750)
tensor(0.9375)
tensor(0.8750)
tensor(0.9375)
tensor(0.9375)
tensor(0.8125)
tensor(0.3750)
tensor(0.9375)
tensor(0.8125)
tensor(0.3750)
tensor(0.7500)
tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.8750)
tensor(0.9375)
tensor(0.9375)
tensor(0.8125)
tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.8125)
tensor(0.9375)
tensor(0.9375)
tensor(0.6875)
tensor(0.8750)
tensor(0.9375)
tensor(0.9375)
tensor(0.4375)
tensor(0.8750)
tensor(0.8125)
tensor(0.8750)
tensor(0.8125)
tensor(0.9375)
tensor(0.8750)
tensor(0.9375)
tensor(0.3750)
tensor(0.9375)
tensor(0.6250)
tensor(0.9375)
tensor(0.0625)
tensor(0.9375)
tensor(0.9375)
tensor(0.8750)
tensor(0.2500)
tensor(0.8750)
tensor(0.9375)
tensor(0.5000)
tensor(0.8125)
tensor(0.7500)
tensor(0.9375)
tensor(0.6875)
tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.9375)
tensor(0.8125)
tensor(0.9375)
tensor(0.9375)
tensor(0.8125)
tensor(0.9375)
tensor(0.9

In [17]:
from rect_prune_ood.lib.eval.metrics import *

#openimage_o_scores = np.load(os.path.join(results_dir, 'imagenet', 'apesp@95', 'openimage-o.npy'), allow_pickle=True).item()["energy_score"]

all_results = []

for i in range(2):#len(imagenet_c_pertubations)):

    imagenet_scores = np.load(os.path.join(ablation_study_results_dir, f"ablation_study_pertubations_imagenet_{i+1}.npy"), allow_pickle=True).item()["energy_score"]
    openimage_o_scores = np.load(os.path.join(ablation_study_results_dir, f"ablation_study_pertubations_openimage-o_{i+1}.npy"), allow_pickle=True).item()["energy_score"]

    results = cal_metric(imagenet_scores, openimage_o_scores)
    results["num_pertubations"] = i + 1

    all_results.append(results)

print(all_results)

print("num_p   FPR   AUROC", end='')

for results in all_results:
    print()
    print('{value:5d}'.format(value=results['num_pertubations']), end='')
    print(' {value:6.2f}'.format(value=results['FPR']), end='')
    print('{value:6.2f}'.format(value=results['AUROC']), end='')

[{'FPR': 0.615358439201452, 'AUROC': 0.8491842842558984, 'DTERR': 0.19166962384669978, 'AUIN': 0.9391623413320852, 'AUOUT': 0.6689227795112996, 'num_pertubations': 1}, {'FPR': 0.6607872050816697, 'AUROC': 0.842156749092559, 'DTERR': 0.20289212207239177, 'AUIN': 0.9374953932022745, 'AUOUT': 0.6450837350030604, 'num_pertubations': 2}]
num_p   FPR   AUROC
    1   0.62  0.85
    2   0.66  0.84

In [ ]:
react_imagenet_scores = np.load(os.path.join(results_dir, 'imagenet', 'react', 'imagenet.npy'), allow_pickle=True).item()["energy_score"]
react_places_scores = np.load(os.path.join(results_dir, 'imagenet', 'react', 'places.npy'), allow_pickle=True).item()["energy_score"]